# Python: Instrumental Variables Estimations

---

## Introduction

In this notebook we will demonstrate how to use instrumental variables (IV) estimation (or better Two-Stage-Least Squares, 2SLS) to estimate the parameters in a linear regression model. If you want some more theoretical background on why we may need to use these techniques you may want to refer to any decent Econometrics textbook.

Here we will be very short on the problem setup and big on the implementation! When you estimate a linear regression model, say

$$y = \alpha_0 + \alpha_1 x_1 + \alpha_2 x_2 + \alpha_3 x_3 + u$$

the most crucial of all assumptions you got to make is that the explanatory variables $x_1$ to $x_3$ are uncorrelated to the error term $u$. Of course, the error term $u$ is unobservable and hence it is impossible to empirically test this assumption (notwithstanding a related test introduced below) and you ought to think very carefully whether there may be any reason that makes it likely that this assumption might be breached. Seasoned econometricians would immediately rattle down simultaneity bias, measurement error and omitted relevant variables as the three causes for this to happen.

In some such situations you can actually fix the problem, e.g. by including additional explanatory variables into the model, but in others that is impossible and you need to accept that there is a high probability that, say, $x_3$ is correlated with $u$. We would then call $x_3$ an endogenous variable and all those explanatory variables that do not have that issue are called exogenous. If you still persist with estimating that model by Ordinary Least Squares (OLS) you will have to accept that your estimated coefficients come from a random distribution that on average will not produce the correct (yet unknown) value, in technical lingo, the estimators are biased.

To the rescue come instrumental variables (IV) estimation. What we need to use this technique is what is called an instrumental variable. And if only $x_3$ is potentially correlated with the error term we need at least one such variable, but more could be useful as well. You always need at least as many instruments as you have endogenous variables. These instruments need to have the following crucial properties, they need to be correlated to the endogenous variable, uncorrelated to the error term and shouldn't be part of the model explaining the dependent variable $y$.

## The basic idea of IV/2SLS

Here is a brief outline of what happens when you use IV, in the form of a 2SLS regression.

1. Take all of your endogenous variables and run regressions with these as the dependent variable and all other exogenous and all instrumental variables as explanatory variables. From these regressions you get the predicted values for all your endogenous variables, e.g. $\hat{x}_3$. These regression(s) are called first stage regressions. The idea is that, as all explanatory variables in this first stage regression are assumed to be uncorrelated with the error term, the variable $\hat{x}_3$ is also uncorrelated with the unobserved error term $u$. All variation in $x_3$ that was correlated with the error term $u$ must have ended up in the error term of this auxiliary regression.

2. In the second stage of the procedure we return to our original regression model and replace $x_3$ with $\hat{x}_3$ and then estimate values for the parameters $\alpha_0$ to $\alpha_3$ by OLS. These estimators, at least for sufficiently large samples and sufficiently large correlation of the instruments with endogenous variables, will deliver unbiased estimates (technical lingo: consistent).

This sounds pretty easy. There is a slight complication, the standard errors that the second stage OLS regression delivers are incorrect and we need to calculate different standard errors. But that will happen automatically in the procedure below.

## Implementation in Python

The main Python packages we'll use are:
- `linearmodels` for IV regression (`IV2SLS` class)
- `pandas` for data handling
- `statsmodels` for OLS and statistical tests
- `numpy` for numerical operations

If you use IV a lot in your work, you may want to create convenient wrapper functions. But if you are applying IV for the first time it is actually very instructive to go through some of the steps in a bit more detail. It is also good to see that really there is not a lot of technical magic... just a great idea!

In [31]:
# Import required libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# For nice output formatting
pd.set_option('display.precision', 4)
np.set_printoptions(precision=4, suppress=True)

## Example

We will use the Women's Wages dataset to illustrate the use of the IV regression ([mroz.csv](https://github.com/datasquad/ECLR/blob/gh-pages/data/Mroz.csv)). The dependent variable which we use here is the log wage `lwage` and we are interested in whether the years of education (`educ`) has a positive influence on this log wage (here we mirror the analysis in Wooldridge's Example 15.1 in his Introductory Econometrics textbook).

Let's first import the data:

In [32]:
# Load the data
# Note: You'll need to adjust the path to where your data is located
mydata = pd.read_csv('data/mroz.csv', na_values='.')

# Alternative if data is in current directory:
# mydata = pd.read_csv('mroz.csv', na_values='.')

Remove all observations with missing wages from the dataframe:

In [33]:
# Remove observations with missing wages
mydata = mydata[mydata['wage'].notna()]

# Or using the query method:
# mydata = mydata.query('wage.notna()', engine='python')

print(f"Dataset shape: {mydata.shape}")
print(f"\nFirst few rows:")
mydata.head()

Dataset shape: (428, 22)

First few rows:


,inlf,hours,kidslt6,kidsge6,age,educ,wage,repwage,hushrs,husage,...,faminc,mtr,motheduc,fatheduc,unem,city,exper,nwifeinc,lwage,expersq
0,1,1610,1,0,32,12,3.3540,2.65,2708,34,...,16310,0.7215,12,7,5.0,0,14,10.9101,1.2102,196
1,1,1656,0,2,30,12,1.3889,2.65,2310,30,...,21800,0.6615,7,7,11.0,1,5,19.5000,0.3285,25
2,1,1980,1,3,35,12,4.5455,4.04,3072,40,...,21040,0.6915,12,7,5.0,0,15,12.0399,1.5141,225
3,1,456,0,3,34,12,1.0965,3.25,1920,53,...,7300,0.7815,7,7,5.0,0,6,6.8000,0.0921,36
4,1,1568,1,2,31,14,4.5918,3.60,2000,32,...,27300,0.6215,12,14,9.5,1,7,20.1001,1.5243,49


An extremely simple model would be to estimate the following OLS regression which models lwage as a function of a constant and educ.

In [34]:
# OLS estimation: lwage ~ educ
reg_ex0 = smf.ols('lwage ~ educ', data=mydata).fit()

print("="*70)
print("OLS Estimation")
print("="*70)
print(reg_ex0.summary())

OLS Estimation
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     56.93
Date:                Sun, 28 Dec 2025   Prob (F-statistic):           2.76e-13
Time:                        16:06:17   Log-Likelihood:                -441.26
No. Observations:                 428   AIC:                             886.5
Df Residuals:                     426   BIC:                             894.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1852      0.185     

This seems to indicate that every additional year of education increases the wage by almost 11% (recall the interpretation of a coefficient in a log-lin model!). The issue with this sort of model is that education is most likely to be correlated with individual characteristics that are important for the person's wage, but not modeled (and hence captured by the error term).

What we need is an instrument that meets the conditions outlined above and here and as in Wooldridge's example we use the father's education as an instrument. The way to do this is as follows:

In [35]:
# IV estimation using father's education as instrument
# Dependent variable: lwage
# Endogenous variable: educ
# Instrument: fatheduc

# Prepare data for IV2SLS
from linearmodels.iv import IV2SLS

# Using the formula interface
reg_iv0 = IV2SLS.from_formula('lwage ~ 1 + [educ ~ fatheduc]', data=mydata).fit(cov_type='unadjusted')

print("="*70)
print("IV Estimation")
print("="*70)
print(reg_iv0.summary)

IV Estimation
                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.0934
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0913
No. Observations:                 428   F-statistic:                    2.8487
Date:                Sun, Dec 28 2025   P-value (F-stat)                0.0915
Time:                        16:06:17   Distribution:                  chi2(1)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      0.4411     0.4451     0.9911     0.3216     -0.4312      1.3134
educ           0.0592     0.0351     1

The `IV2SLS` function works similarly to the `ols` function but uses a special formula syntax. The endogenous variable is specified in square brackets: `[educ ~ fatheduc]` means "instrument educ with fatheduc".

Clearly, the effect of an additional year of education has significantly dropped and is now only marginally significant. It is, of course, often a feature of IV estimation that the estimated standard errors are significantly larger than those of the OLS estimators. The size of the standard error depends a lot on the strength of the relation between the endogenous explanatory variables which can be checked by looking at the R² of the regression of educ on fatheduc.

In [36]:
# Check first stage R-squared
first_stage_check = smf.ols('educ ~ fatheduc', data=mydata).fit()
print(f"First stage R-squared: {first_stage_check.rsquared:.4f}")

First stage R-squared: 0.1726


In order to illustrate the full functionality of the IV procedure we re-estimate the model with extra explanatory variables and more instruments than endogenous variables which means that really we are applying a 2SLS estimation (This is the example estimated in Wooldridge's Example 15.5). Let's start by estimating this model by OLS (as we need this result later).

In [37]:
# OLS estimation with additional controls
reg_1 = smf.ols('lwage ~ educ + age + exper + expersq', data=mydata).fit()

print("="*70)
print("OLS Estimation")
print("="*70)
print(reg_1.summary())

OLS Estimation
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.157
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     19.67
Date:                Sun, 28 Dec 2025   Prob (F-statistic):           7.33e-15
Time:                        16:06:17   Log-Likelihood:                -431.60
No. Observations:                 428   AIC:                             873.2
Df Residuals:                     423   BIC:                             893.5
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.5334      0.278     

The estimated coefficient for educ is approximately 0.108 with standard error 0.014. Then we estimate the 2SLS regression with `fatheduc` and `motheduc` as instruments.

In [38]:
# IV estimation with multiple instruments
# Dependent: lwage
# Endogenous: educ
# Instruments: fatheduc, motheduc
# Exogenous: age, exper, expersq

reg_iv1 = IV2SLS.from_formula(
    'lwage ~ 1 + age + exper + expersq + [educ ~ fatheduc + motheduc]',
    data=mydata
).fit(cov_type='unadjusted')

print("="*70)
print("IV Estimation with Multiple Instruments")
print("="*70)
print(reg_iv1.summary)

IV Estimation with Multiple Instruments
                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.1353
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1272
No. Observations:                 428   F-statistic:                    24.628
Date:                Sun, Dec 28 2025   P-value (F-stat)                0.0001
Time:                        16:06:17   Distribution:                  chi2(4)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      0.0667     0.4564     0.1462     0.8838     -0.8278      0.9613
age         

The formula syntax `[educ ~ fatheduc + motheduc]` tells Python:
- `educ` is the endogenous variable to be instrumented
- `fatheduc + motheduc` are the instruments
- Variables outside the brackets (`age + exper + expersq`) are treated as exogenous

Let's create a comparison table of OLS vs IV results:

In [39]:
# Create comparison table
def create_comparison_table(ols_model, iv_model, var_names):
    """
    Create a comparison table for OLS and IV estimates
    """
    comparison = pd.DataFrame({
        'OLS Coef': ols_model.params[var_names],
        'OLS SE': ols_model.bse[var_names],
        'IV Coef': iv_model.params[var_names],
        'IV SE': iv_model.std_errors[var_names]
    })
    
    # Add significance stars
    ols_pvals = ols_model.pvalues[var_names]
    iv_pvals = iv_model.pvalues[var_names]
    
    def get_stars(pval):
        if pval < 0.01:
            return '***'
        elif pval < 0.05:
            return '**'
        elif pval < 0.10:
            return '*'
        return ''
    
    comparison['OLS Sig'] = [get_stars(p) for p in ols_pvals]
    comparison['IV Sig'] = [get_stars(p) for p in iv_pvals]
    
    return comparison

# Variables to compare
vars_to_compare = ['Intercept', 'educ', 'age', 'exper', 'expersq']

print("\nComparison of OLS and IV Estimates")
print("="*70)
comparison_df = create_comparison_table(reg_1, reg_iv1, vars_to_compare)
print(comparison_df)
print("\nSignificance: *** p<0.01, ** p<0.05, * p<0.10")
print(f"\nOLS R-squared: {reg_1.rsquared:.4f}")
print(f"IV R-squared: {reg_iv1.rsquared:.4f}")


Comparison of OLS and IV Estimates
           OLS Coef  OLS SE  IV Coef   IV SE OLS Sig IV Sig
Intercept   -0.5334  0.2778   0.0667  0.4564       *       
educ         0.1075  0.0142   0.0610  0.0314     ***      *
age          0.0003  0.0049  -0.0004  0.0049               
exper        0.0416  0.0132   0.0442  0.0134     ***    ***
expersq     -0.0008  0.0004  -0.0009  0.0004      **     **

Significance: *** p<0.01, ** p<0.05, * p<0.10

OLS R-squared: 0.1568
IV R-squared: 0.1353


## IV Related Testing Procedures

One feature of IV estimations is that in general it is an inferior estimator of $\beta$ if all explanatory variables are exogenous. In that case, assuming that all other Gauss-Markov assumptions are met, the OLS estimator is the BLUE estimator. In other words, IV estimators have larger standard errors for the coefficient estimates. Therefore, one would really like to avoid having to rely on IV estimators, unless, of course, they are the only estimators that deliver consistent estimates.

So there are usually three tests one performs in this context:
1. Test to examine that the chosen instruments are indeed sufficiently strongly correlated to the endogenous variable (Instrument relevance)
2. Whether the potentially endogenous variable is indeed endogenous (Testing for exogeneity)
3. That the instruments are indeed exogenous (Sargan test)

### Instrument Relevance

The entire 2SLS procedure hinges on the instruments chosen being useful instruments. Useful here means that they are sufficiently strongly correlated to the endogenous variable.

We can use the first stage regression (described in the Introduction) to test whether that is indeed the case. So here is the first stage regression:

In [40]:
# First Stage Regression
reg_fs1 = smf.ols('educ ~ age + exper + expersq + fatheduc + motheduc', data=mydata).fit()

print("First Stage Regression Summary:")
print(reg_fs1.summary())

First Stage Regression Summary:
                            OLS Regression Results                            
Dep. Variable:                   educ   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.202
Method:                 Least Squares   F-statistic:                     22.67
Date:                Sun, 28 Dec 2025   Prob (F-statistic):           3.64e-20
Time:                        16:06:18   Log-Likelihood:                -909.64
No. Observations:                 428   AIC:                             1831.
Df Residuals:                     422   BIC:                             1856.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      8.851

What we now need to know is whether the instruments `fatheduc` and `motheduc` explain a sufficient amount of variation in `educ`. We can use a standard F-test to test this. We are basically testing the null hypothesis that the coefficients to `fatheduc` and `motheduc` are equal to 0.

In [41]:
# F-test for instrument relevance
# H0: fatheduc = 0 AND motheduc = 0

# Define the hypothesis matrix
hypotheses = 'fatheduc = 0, motheduc = 0'

# Perform F-test
f_test = reg_fs1.f_test(hypotheses)

print("="*70)
print("F-test for Instrument Relevance")
print("="*70)
print(f"H0: fatheduc = 0 AND motheduc = 0")
print(f"\nF-statistic: {f_test.fvalue:.3f}")
print(f"p-value: {f_test.pvalue:.6f}")
print(f"Degrees of freedom: ({f_test.df_num:.0f}, {f_test.df_denom:.0f})")

if f_test.pvalue < 0.05:
    print("\nConclusion: Reject H0 - Instruments are relevant (strong instruments)")
else:
    print("\nConclusion: Cannot reject H0 - Weak instruments problem")

F-test for Instrument Relevance
H0: fatheduc = 0 AND motheduc = 0

F-statistic: 54.943
p-value: 0.000000
Degrees of freedom: (2, 422)

Conclusion: Reject H0 - Instruments are relevant (strong instruments)


The value of the F-test is very high with an extremely low p-value. So in this case we can clearly reject the null hypothesis that the instruments are irrelevant.

### Testing for Exogeneity

You really only want to use IV/2SLS if you are really dealing with endogenous explanatory variables. If the variable you suspected wasn't endogenous, then IV only has disadvantages compared to OLS. Most crucially it will deliver much larger standard errors. For this reason you really want to make sure that you do have an endogeneity problem.

The celebrated test to use in this case is the Hausman test. Here we use a slightly different implementation to the original Hausman test, the so-called Hausman-Wu test.

In the end it is pretty straightforward and you only need simple regressions to implement it. In a first step you run the first step regression(s) of the 2SLS procedure, which we did earlier and saved the results in `reg_fs1`. In a second step you add the residual(s) from this first step into the original model:

In [42]:
# Hausman-Wu test for exogeneity
# Step 1: Get residuals from first stage (already computed above)
# Step 2: Add residuals to original model

# Create a copy of the data with first stage residuals
mydata_test = mydata.copy()
mydata_test['fs_residuals'] = reg_fs1.resid

# Run augmented regression
reg_HSW1 = smf.ols('lwage ~ educ + age + exper + expersq + fs_residuals', 
                    data=mydata_test).fit()

print("Hausman-Wu Test Regression:")
print(reg_HSW1.summary())

Hausman-Wu Test Regression:
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.162
Model:                            OLS   Adj. R-squared:                  0.152
Method:                 Least Squares   F-statistic:                     16.37
Date:                Sun, 28 Dec 2025   Prob (F-statistic):           9.06e-15
Time:                        16:06:18   Log-Likelihood:                -430.17
No. Observations:                 428   AIC:                             872.3
Df Residuals:                     422   BIC:                             896.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.0

Now we need to compare this result to the one we got from the original model `reg_1`. If `educ` is indeed endogenous, then the first stage regression should have isolated the variation of `educ` that was correlated with error term in the residual of the first stage regression. In that case the included `fs_residuals` should be relevant. As there may potentially be more than one endogenous variable and hence more than one first stage residual, we use an F-test to test the null hypothesis that these residuals are irrelevant (and hence endogeneity not being a problem).

In [43]:
# F-test for exogeneity
# H0: fs_residuals = 0 (educ is exogenous)

exog_test = reg_HSW1.f_test('fs_residuals = 0')

print("="*70)
print("Hausman-Wu Test for Exogeneity")
print("="*70)
print(f"H0: educ is exogenous (fs_residuals = 0)")
print(f"\nF-statistic: {exog_test.fvalue:.3f}")
print(f"p-value: {exog_test.pvalue:.4f}")

print("\nInterpretation:")
if exog_test.pvalue < 0.05:
    print(f"At α = 0.05: Reject H0 - educ is endogenous, use IV")
else:
    print(f"At α = 0.05: Cannot reject H0 - educ may be exogenous")
    
if exog_test.pvalue < 0.10:
    print(f"At α = 0.10: Reject H0 - educ is endogenous, use IV")
else:
    print(f"At α = 0.10: Cannot reject H0 - educ may be exogenous")

Hausman-Wu Test for Exogeneity
H0: educ is exogenous (fs_residuals = 0)

F-statistic: 2.818
p-value: 0.0940

Interpretation:
At α = 0.05: Cannot reject H0 - educ may be exogenous
At α = 0.10: Reject H0 - educ is endogenous, use IV


The result shows a p-value around 0.094. So at an $\alpha = 0.05$ we just fail to reject the null of `educ` being exogenous, however, at an $\alpha = 0.10$ we would reject exogeneity of `educ`. So the case for using IV or OLS is not clearcut here.

### Sargan Test for Instrument Validity

One crucial property of instruments is that they ought to be uncorrelated to the regression error terms $u$. Instrument exogeneity is set as the null hypothesis of this following test with the alternative hypothesis being that the instruments are endogenous. This test can only be applied if you have more instruments than endogenous variables. It is therefore sometimes also called the test for overidentifying restrictions.

The test is rather simple to implement. Take the residuals from the 2SLS regression `reg_iv1.resids` and use them as the dependent variable in a new regression in which you regress them on all exogenous explanatory variables and all instruments.

In [44]:
# Sargan test for overidentifying restrictions
# Step 1: Get residuals from IV regression
mydata_sargan = mydata.copy()
mydata_sargan['iv_residuals'] = reg_iv1.resids.values

# Step 2: Regress IV residuals on all exogenous variables and instruments
reg_sargan1 = smf.ols(
    'iv_residuals ~ age + exper + expersq + fatheduc + motheduc',
    data=mydata_sargan
).fit()

print("Sargan Test Regression:")
print(reg_sargan1.summary())

Sargan Test Regression:
                            OLS Regression Results                            
Dep. Variable:           iv_residuals   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.011
Method:                 Least Squares   F-statistic:                   0.07937
Date:                Sun, 28 Dec 2025   Prob (F-statistic):              0.995
Time:                        16:06:18   Log-Likelihood:                -436.78
No. Observations:                 428   AIC:                             885.6
Df Residuals:                     422   BIC:                             909.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0316      0

If the instruments are valid (null hypothesis), they should be uncorrelated to these residuals and hence we apply the following $\chi^2$ test. We use the $R^2$ of this regression and calculate $n \times R^2$.

In [45]:
# Calculate Sargan test statistic
n = len(mydata_sargan)
Sargan_test = reg_sargan1.rsquared * n

# Degrees of freedom = number of instruments - number of endogenous variables
# We have 2 instruments (fatheduc, motheduc) and 1 endogenous variable (educ)
df = 2 - 1  # = 1

# Calculate p-value from chi-squared distribution
p_value = 1 - stats.chi2.cdf(Sargan_test, df)

print("="*70)
print("Sargan Test for Instrument Validity")
print("="*70)
print(f"H0: Instruments are valid (exogenous)")
print(f"\nTest statistic (n × R²): {Sargan_test:.4f}")
print(f"Degrees of freedom: {df}")
print(f"p-value: {p_value:.4f}")

if p_value > 0.05:
    print("\nConclusion: Cannot reject H0 - Instruments appear to be valid")
else:
    print("\nConclusion: Reject H0 - Instruments may not be valid")

Sargan Test for Instrument Validity
H0: Instruments are valid (exogenous)

Test statistic (n × R²): 0.4021
Degrees of freedom: 1
p-value: 0.5260

Conclusion: Cannot reject H0 - Instruments appear to be valid


We find that the p-value of this test is approximately 0.526 and hence we do not reject the null hypothesis of instrument validity. The p-value was obtained from a $\chi^2$ distribution with one degree of freedom. That was one here as we had two instruments for one endogenous variable (2-1) or one overidentifying restriction.

## Summary

In this notebook we have covered:

1. **Basic IV/2SLS estimation** using the `linearmodels` package
2. **Instrument relevance testing** - checking if instruments are sufficiently correlated with endogenous variables
3. **Hausman-Wu test** - testing whether suspected endogenous variables are actually endogenous
4. **Sargan test** - testing the validity of overidentifying restrictions

### Key Python Functions Used:

- `IV2SLS.from_formula()` - Main IV regression function
- `smf.ols()` - OLS regression for comparison and auxiliary tests
- `.f_test()` - F-tests for joint hypothesis testing
- `stats.chi2.cdf()` - Chi-squared distribution for Sargan test

### Important Notes:

- The formula syntax for IV2SLS uses square brackets: `[endogenous ~ instruments]`
- Variables outside the brackets are treated as exogenous
- IV standard errors are automatically corrected (unlike manual 2-stage OLS)
- Always check instrument relevance before interpreting IV results